# Video features on Colab

Downloads an OSSL-v2 subset and extracts DINOv2 and V-JEPA 2 tokens on a Colab GPU, then copies
the feature cache (about 1 GB) to Drive. Train afterwards on any machine from that cache.

Before running: *Runtime > Change runtime type > T4 GPU*, and a Colab secret named
`HF_TOKEN` (key icon on the left) holding a Hugging Face *read* token.

In [ ]:
# Subset size: keep the same numbers everywhere (download, extraction, targets, training).
N_TRAIN_MOVIES = 400
N_TEST_MOVIES = 100
DRIVE_DIR = "/content/drive/MyDrive/scene-to-score"
COPY_AUDIO = True  # keep the wav files on Drive for the CLAP targets (about 25 GB)

In [ ]:
import shutil
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
free_gb = shutil.disk_usage("/content").free / 1e9
print(f"free disk: {free_gb:.0f} GB")
needed = 0.135 * (N_TRAIN_MOVIES + N_TEST_MOVIES) + 10
assert free_gb > needed, f"need about {needed:.0f} GB: lower N_TRAIN_MOVIES / N_TEST_MOVIES"

In [ ]:
import os

from google.colab import drive, userdata

drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["STS_DATA_DIR"] = "/content/data"

In [ ]:
%cd /content
!git clone -q https://github.com/ademasles/scene-to-score.git
%cd /content/scene-to-score
!pip -q install uv
!uv sync --no-dev

The lock file pins PyTorch for CUDA 13.0. If the next cell prints `False`, the Colab driver is
older: reinstall PyTorch for CUDA 12.8 with the cell after it.

In [ ]:
!uv run --no-sync python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

In [ ]:
# Only if the cell above printed False.
!uv pip install -q --reinstall torch --index-url https://download.pytorch.org/whl/cu128
!uv run --no-sync python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

In [ ]:
SUBSET = f"data.n_train_movies={N_TRAIN_MOVIES} data.n_test_movies={N_TEST_MOVIES}"
!uv run --no-sync python -m scene_to_score.data.download {SUBSET}

In [ ]:
!uv run --no-sync python -m scene_to_score.features.video encoder=dinov2-large {SUBSET}
!uv run --no-sync python -m scene_to_score.features.video encoder=vjepa2-large {SUBSET}

In [ ]:
import shutil
from pathlib import Path

data, target = Path("/content/data"), Path(DRIVE_DIR)
shutil.copytree(data / "features", target / "features", dirs_exist_ok=True)
if COPY_AUDIO:  # wav files and the index, not the mp4s

    def keep(d, names):
        wanted = (".wav", ".csv", ".txt", ".md")
        return [n for n in names if not (Path(d, n).is_dir() or n.endswith(wanted))]

    shutil.copytree(data / "ossl-v2", target / "ossl-v2", ignore=keep, dirs_exist_ok=True)
!du -sh {DRIVE_DIR}/*